# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to explore and process the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
 
`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
from pprint import pprint

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print high-level metadata information
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}\n")

# Display key metadata fields
print(f"License: {metadata.license}")
print(f"Identifier: {getattr(metadata, 'identifier', 'N/A')}")
print(f"Data collection timeframe: {getattr(metadata, 'dataCollectionTimeframe', 'N/A')}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview
Review available record sets, their `@id`s, and fields within the dataset.

In [ ]:
# List all record sets and fields by their `@id`
record_sets = list(dataset.record_sets)

if not record_sets:
    print("No record sets found in this dataset. Please verify with dataset creators.")
else:
    print(f"Found {len(record_sets)} record sets:")
    for rs in record_sets:
        print(f"- Record set name: {rs.name} | @id: {rs.id}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - Field '{field.name}' (@id: {field.id}) | dataType: {field.data_type}")
        print("")

# Store record set @ids for later
record_set_ids = [rs.id for rs in record_sets]
# For demonstration, pick the first record set for further exploration (update if necessary)
chosen_record_set_id = record_set_ids[0] if record_set_ids else None

## 3. Data Extraction
Load data from a specific record set (using the record set and field `@id` values found above) into a DataFrame for analysis.

In [ ]:
# Extract data from each record set and load into pandas DataFrames
dataframes = {}

if not record_set_ids:
    print("No record sets available to extract records from.")
else:
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set @id: {record_set_id}")
            print(f"Columns: {df.columns.tolist()}\n")
        except Exception as e:
            print(f"Error loading record set {record_set_id}: {e}")

# Preview the first record set (as example, change chosen_record_set_id from cell above as needed)
if chosen_record_set_id and chosen_record_set_id in dataframes:
    print(f"Columns for record set {chosen_record_set_id}:")
    print(dataframes[chosen_record_set_id].columns.tolist())
    dataframes[chosen_record_set_id].head()
else:
    print("No valid DataFrame loaded for preview.")

## 4. Exploratory Data Analysis (EDA)
Apply common preprocessing: filtering, normalization, and group-by operations. (Replace `<numeric_field_id>` and `<group_field_id>` with appropriate `@id`s for your record set as needed.)

In [ ]:
# Example EDA: Filtering, normalization, and grouping

# Replace these ids with valid field @id strings from earlier display for a numeric and a grouping field
record_set_id = chosen_record_set_id  # e.g., the @id string from the data overview

# Show columns (field @ids) again for convenience
if record_set_id and record_set_id in dataframes:
    df = dataframes[record_set_id]
    print("Available columns (@id):", df.columns.tolist())

    # Select numeric field by @id (replace this with a real one for your dataset!)
    numeric_field = None
    group_field = None
    # Example: search for first numeric-like field
    for col in df.columns:
        if df[col].dtype in ['float64', 'int64']:
            numeric_field = col
            break
    # Example: first non-numeric as group field
    for col in df.columns:
        if df[col] != numeric_field and df[col].dtype == 'object':
            group_field = col
            break

    if numeric_field:
        threshold = df[numeric_field].mean()  # Use the mean as threshold for illustration
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped mean {numeric_field} by {group_field}:")
            print(grouped_df.head())
        else:
            print("No suitable group field detected.")
    else:
        print("No numeric field found for EDA in this record set.")
else:
    print("No valid DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields. (This cell will not plot unless you have numeric/group fields above; update accordingly.)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If you have a numeric and group field, visualize their relationship
if record_set_id and record_set_id in dataframes and numeric_field and group_field:
    plt.figure(figsize=(8, 5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"Distribution of {numeric_field} by {group_field}")
    plt.xticks(rotation=40)
    plt.tight_layout()
    plt.show()
elif record_set_id and record_set_id in dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(f"{numeric_field}")
    plt.tight_layout()
    plt.show()
else:
    print("No suitable fields available for plotting.")

## 6. Conclusion
This notebook demonstrated how to load, inspect, and perform basic exploratory analysis on the FAIR² dataset using `mlcroissant`. For further insight, adjust field selections or perform more advanced analysis based on your research interests. 